In [ ]:
# !pip install optuna


   ---------------------------------------- 0/2 [colorlog]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ----

In [3]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    #  using normal 
    # model.fit(X_train,y_train)
    # y_pred = model.predict(X_test)
    # accuracy = accuracy_score(y_test,y_pred)

    return score  # Return the accuracy score for Optuna to maximize
    # return accuracy

In [13]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-06-27 22:00:18,831] A new study created in memory with name: no-name-7010d99a-1b25-4084-a638-9df612ed7f44
[I 2025-06-27 22:00:19,321] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 82, 'max_depth': 16}. Best is trial 0 with value: 0.7635009310986964.
[I 2025-06-27 22:00:19,727] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 67, 'max_depth': 11}. Best is trial 1 with value: 0.7653631284916201.
[I 2025-06-27 22:00:20,222] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 102, 'max_depth': 11}. Best is trial 1 with value: 0.7653631284916201.
[I 2025-06-27 22:00:20,976] Trial 3 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 151, 'max_depth': 20}. Best is trial 3 with value: 0.7728119180633147.
[I 2025-06-27 22:00:21,810] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 183, 'max_depth': 15}. Best is trial 3 with value: 0.772811

In [14]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350094
Best hyperparameters: {'n_estimators': 70, 'max_depth': 18}


In [15]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


In [16]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [25]:
# 1. Optimization History
plot_optimization_history(study).show()

In [18]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [19]:
# 3. Slice Plot
plot_slice(study).show()

In [24]:
# 4. Contour Plot
plot_contour(study).show()

In [23]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [22]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [26]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [27]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-06-27 22:34:51,321] A new study created in memory with name: no-name-4d436ea2-ebdf-48a4-aead-d125b7f50158
[I 2025-06-27 22:34:52,283] Trial 0 finished with value: 0.7523277467411545 and parameters: {'classifier': 'RandomForest', 'n_estimators': 240, 'max_depth': 3, 'min_samples_split': 7, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 0 with value: 0.7523277467411545.
[I 2025-06-27 22:34:52,552] Trial 1 finished with value: 0.7635009310986964 and parameters: {'classifier': 'RandomForest', 'n_estimators': 61, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 1, 'bootstrap': False}. Best is trial 1 with value: 0.7635009310986964.
[I 2025-06-27 22:34:52,573] Trial 2 finished with value: 0.7858472998137803 and parameters: {'classifier': 'SVM', 'C': 0.6537295803774336, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 2 with value: 0.7858472998137803.
[I 2025-06-27 22:34:53,519] Trial 3 finished with value: 0.7486033519553073 and parameters: {'classifier': 'G

In [28]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.1286687560841135, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [29]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.752328,2025-06-27 22:34:51.322805,2025-06-27 22:34:52.283946,0 days 00:00:00.961141,NaN,False,RandomForest,NaN,NaN,NaN,3.0,2.0,7.0,240.0,COMPLETE
1,1,0.763501,2025-06-27 22:34:52.284578,2025-06-27 22:34:52.551960,0 days 00:00:00.267382,NaN,False,RandomForest,NaN,NaN,NaN,19.0,1.0,6.0,61.0,COMPLETE
2,2,0.785847,2025-06-27 22:34:52.552582,2025-06-27 22:34:52.573683,0 days 00:00:00.021101,0.653730,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.748603,2025-06-27 22:34:52.574509,2025-06-27 22:34:53.519868,0 days 00:00:00.945359,NaN,NaN,GradientBoosting,NaN,NaN,0.148016,14.0,6.0,9.0,72.0,COMPLETE
4,4,0.772812,2025-06-27 22:34:53.520738,2025-06-27 22:34:54.374957,0 days 00:00:00.854219,NaN,False,RandomForest,NaN,NaN,NaN,10.0,4.0,10.0,170.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.787709,2025-06-27 22:35:16.347870,2025-06-27 22:35:16.369959,0 days 00:00:00.022089,0.172820,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.728119,2025-06-27 22:35:16.370491,2025-06-27 22:35:18.362057,0 days 00:00:01.991566,NaN,NaN,GradientBoosting,NaN,NaN,0.226980,16.0,7.0,9.0,186.0,COMPLETE
97,97,0.769088,2025-06-27 22:35:18.362589,2025-06-27 22:35:18.392593,0 days 00:00:00.030004,0.219586,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.785847,2025-06-27 22:35:18.393302,2025-06-27 22:35:18.452872,0 days 00:00:00.059570,13.050283,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [30]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 78
RandomForest        12
GradientBoosting    10
Name: count, dtype: int64

In [31]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.748231
RandomForest        0.763966
SVM                 0.775462
Name: value, dtype: float64

In [32]:
# 1. Optimization History
plot_optimization_history(study).show()

In [33]:
# 3. Slice Plot
plot_slice(study).show()

In [34]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [39]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

[I 2025-06-27 22:41:23,126] A new study created in memory with name: no-name-61c9d2f2-3067-4b77-aaa2-95660dfc08ae


[0]	train-mlogloss:0.85256	eval-mlogloss:0.83854
[1]	train-mlogloss:0.75311	eval-mlogloss:0.73215
[2]	train-mlogloss:0.61918	eval-mlogloss:0.58824
[3]	train-mlogloss:0.50351	eval-mlogloss:0.47001
[4]	train-mlogloss:0.42179	eval-mlogloss:0.38616
[5]	train-mlogloss:0.35290	eval-mlogloss:0.31175
[6]	train-mlogloss:0.30099	eval-mlogloss:0.25590
[7]	train-mlogloss:0.26212	eval-mlogloss:0.21063
[8]	train-mlogloss:0.23140	eval-mlogloss:0.17703
[9]	train-mlogloss:0.20921	eval-mlogloss:0.15140
[10]	train-mlogloss:0.19442	eval-mlogloss:0.14030
[11]	train-mlogloss:0.18677	eval-mlogloss:0.13028
[12]	train-mlogloss:0.17732	eval-mlogloss:0.12010
[13]	train-mlogloss:0.17243	eval-mlogloss:0.11388
[14]	train-mlogloss:0.17131	eval-mlogloss:0.11466
[15]	train-mlogloss:0.17088	eval-mlogloss:0.11457
[16]	train-mlogloss:0.16952	eval-mlogloss:0.11293
[17]	train-mlogloss:0.16576	eval-mlogloss:0.10686
[18]	train-mlogloss:0.16494	eval-mlogloss:0.10739
[19]	train-mlogloss:0.16071	eval-mlogloss:0.10292
[20]	train

[I 2025-06-27 22:41:24,052] Trial 0 finished with value: 1.0 and parameters: {'lambda': 3.8824678601102815e-06, 'alpha': 6.721789534672643e-08, 'eta': 0.2070189698398038, 'gamma': 0.0017096393792721705, 'max_depth': 8, 'min_child_weight': 5, 'subsample': 0.7134122274307897, 'colsample_bytree': 0.554105712470375}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.06150	eval-mlogloss:1.06093
[1]	train-mlogloss:1.02826	eval-mlogloss:1.02331
[2]	train-mlogloss:0.99488	eval-mlogloss:0.98720
[3]	train-mlogloss:0.96045	eval-mlogloss:0.95136
[4]	train-mlogloss:0.92960	eval-mlogloss:0.91955
[5]	train-mlogloss:0.89840	eval-mlogloss:0.88623
[6]	train-mlogloss:0.86923	eval-mlogloss:0.85432
[7]	train-mlogloss:0.84354	eval-mlogloss:0.82864
[8]	train-mlogloss:0.81780	eval-mlogloss:0.80099
[9]	train-mlogloss:0.79198	eval-mlogloss:0.77436
[10]	train-mlogloss:0.76857	eval-mlogloss:0.75047
[11]	train-mlogloss:0.74560	eval-mlogloss:0.72595
[12]	train-mlogloss:0.72343	eval-mlogloss:0.70217
[13]	train-mlogloss:0.70192	eval-mlogloss:0.68009
[14]	train-mlogloss:0.68162	eval-mlogloss:0.65823
[15]	train-mlogloss:0.66186	eval-mlogloss:0.63700
[16]	train-mlogloss:0.64253	eval-mlogloss:0.61636
[17]	train-mlogloss:0.62467	eval-mlogloss:0.59701
[18]	train-mlogloss:0.60740	eval-mlogloss:0.57871
[19]	train-mlogloss:0.59022	eval-mlogloss:0.56076
[20]	train

[I 2025-06-27 22:41:25,569] Trial 1 finished with value: 1.0 and parameters: {'lambda': 1.1304379092842615e-06, 'alpha': 1.8425401519596223e-06, 'eta': 0.029244898807741018, 'gamma': 1.6079247817381406e-05, 'max_depth': 3, 'min_child_weight': 3, 'subsample': 0.4931319796720209, 'colsample_bytree': 0.9593176987688382}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:0.77111	eval-mlogloss:0.75766
[1]	train-mlogloss:0.56841	eval-mlogloss:0.54006


[I 2025-06-27 22:41:33,444] Trial 48 pruned. Trial was pruned at iteration 2.


[0]	train-mlogloss:0.88526	eval-mlogloss:0.88879
[1]	train-mlogloss:0.72352	eval-mlogloss:0.71601


[I 2025-06-27 22:41:33,511] Trial 49 pruned. Trial was pruned at iteration 2.


Best trial: {'lambda': 3.8824678601102815e-06, 'alpha': 6.721789534672643e-08, 'eta': 0.2070189698398038, 'gamma': 0.0017096393792721705, 'max_depth': 8, 'min_child_weight': 5, 'subsample': 0.7134122274307897, 'colsample_bytree': 0.554105712470375}
Best accuracy: 1.0


In [40]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()